# 02 · Profiling, KPIs y semáforo operativo

Este notebook calcula los principales KPIs operativos del proyecto: puntualidad, cancelaciones, retrasos, ocupación, margen y semáforo de rutas.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
REPORTS = PROJECT_ROOT / 'reports'
FIGURES = REPORTS / 'figures'

for p in [DATA_RAW, DATA_PROCESSED, REPORTS, FIGURES]:
    p.mkdir(parents=True, exist_ok=True)

candidate_paths = [
    DATA_RAW / 'ferry_operations_raw.csv',
    DATA_PROCESSED / 'ferry_operations_raw.csv',
    PROJECT_ROOT / 'ferry_operations_raw.csv'
]

csv_path = next((p for p in candidate_paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError('No se encuentra ferry_operations_raw.csv. Ejecuta primero 01_generate_dataset.ipynb o sube el CSV a data/raw/.')

df = pd.read_csv(csv_path)
print('Archivo cargado:', csv_path)
print('Shape:', df.shape)
df.head()


In [ ]:
# Limpieza y tipado
date_cols = ['scheduled_start', 'actual_start', 'scheduled_end', 'actual_end', 'date']
for c in date_cols:
    if c in df.columns:
        df[c] = pd.to_datetime(df[c], errors='coerce')

bool_cols = ['on_time', 'canceled', 'maintenance_flag']
for c in bool_cols:
    if c in df.columns:
        df[c] = df[c].astype(str).str.lower().isin(['true','1','yes','si','sí'])

df['scheduled_date'] = df['scheduled_start'].dt.date
df['hour'] = df['scheduled_start'].dt.hour
df['month'] = df['scheduled_start'].dt.month
df['weekday_num'] = df['scheduled_start'].dt.weekday
df['weekday_name'] = df['scheduled_start'].dt.day_name()
df['completed'] = ~df['canceled']
df['delay_min_clean'] = df['delay_min'].fillna(0)
df['on_time_5'] = (df['delay_min'] <= 5) & df['completed']
df['on_time_15'] = (df['delay_min'] <= 15) & df['completed']

df.to_csv(DATA_PROCESSED / 'ferry_operations_processed.csv', index=False)
print('Dataset procesado guardado en data/processed/ferry_operations_processed.csv')


In [ ]:
# KPIs globales
completed = df[df['completed']]
global_kpis = {
    'total_services': len(df),
    'completed_services': len(completed),
    'canceled_services': int(df['canceled'].sum()),
    'cancel_rate': df['canceled'].mean(),
    'otr_5': completed['on_time_5'].mean(),
    'otr_15': completed['on_time_15'].mean(),
    'avg_delay_min': completed['delay_min'].mean(),
    'p95_delay_min': completed['delay_min'].quantile(0.95),
    'avg_occupancy': df['occupancy'].mean(),
    'avg_margin': df['margin'].mean(),
    'total_margin': df['margin'].sum()
}

global_kpis_df = pd.DataFrame([global_kpis])
global_kpis_df.to_csv(REPORTS / 'global_kpis.csv', index=False)
global_kpis_df


In [ ]:
# KPIs por ruta
route_kpis = (
    df.groupby('route_id')
      .agg(
          services=('operation_id','count'),
          completed_services=('completed','sum'),
          cancel_rate=('canceled','mean'),
          avg_delay_min=('delay_min','mean'),
          p95_delay_min=('delay_min', lambda x: x.dropna().quantile(0.95)),
          otr_5=('on_time_5','mean'),
          otr_15=('on_time_15','mean'),
          avg_occupancy=('occupancy','mean'),
          avg_passengers=('passengers','mean'),
          avg_revenue=('revenue','mean'),
          avg_cost=('total_cost','mean'),
          avg_margin=('margin','mean'),
          total_margin=('margin','sum')
      )
      .reset_index()
)

# Score de priorización: más delay, más P95, peor OTR y menor margen relativo => más prioridad
def minmax(s):
    return (s - s.min()) / (s.max() - s.min()) if s.max() != s.min() else 0

route_kpis['delay_score'] = minmax(route_kpis['avg_delay_min'])
route_kpis['p95_score'] = minmax(route_kpis['p95_delay_min'])
route_kpis['otr_risk'] = 1 - minmax(route_kpis['otr_15'])
route_kpis['margin_risk'] = 1 - minmax(route_kpis['avg_margin'])
route_kpis['balance_score'] = (
    0.35 * route_kpis['delay_score'] +
    0.25 * route_kpis['p95_score'] +
    0.25 * route_kpis['otr_risk'] +
    0.15 * route_kpis['margin_risk']
)

route_kpis['status'] = np.select(
    [route_kpis['balance_score'] >= 0.66, route_kpis['balance_score'] >= 0.40],
    ['AMBER', 'WATCH'],
    default='GREEN'
)

route_kpis = route_kpis.sort_values('balance_score', ascending=False)
route_kpis.to_csv(REPORTS / 'route_kpis.csv', index=False)
route_kpis.head(10)


In [ ]:
# Ranking de rutas con mayor retraso medio
top_delay_routes = route_kpis.sort_values('avg_delay_min', ascending=False)[
    ['route_id','avg_delay_min','p95_delay_min','otr_5','otr_15','avg_margin','status']
]
top_delay_routes.to_csv(REPORTS / 'top_delay_routes.csv', index=False)
top_delay_routes.head(10)


In [ ]:
# Semáforo operativo
control_tower = route_kpis[[
    'route_id','services','completed_services','cancel_rate','avg_delay_min','p95_delay_min',
    'otr_5','otr_15','avg_occupancy','avg_margin','total_margin','balance_score','status'
]].copy()

control_tower['recommended_action'] = np.select(
    [
        control_tower['status'].eq('AMBER'),
        control_tower['status'].eq('WATCH')
    ],
    [
        'Priorizar revisión operativa: puntualidad, congestión, clima y mantenimiento',
        'Monitorizar semanalmente y revisar franjas críticas'
    ],
    default='Mantener seguimiento estándar'
)

control_tower.to_csv(REPORTS / 'control_tower_routes.csv', index=False)
control_tower


In [ ]:
# Resumen ejecutivo en Markdown
top5 = top_delay_routes.head(5).copy()
amber_routes = control_tower[control_tower['status'].eq('AMBER')]['route_id'].tolist()
green_routes = control_tower[control_tower['status'].eq('GREEN')]['route_id'].tolist()

summary = f'''# Executive Summary — Control Operativo y Eficiencia (Ferries)

## KPIs globales
- Servicios totales: **{global_kpis['total_services']:.0f}**
- Servicios completados: **{global_kpis['completed_services']:.0f}**
- Cancel Rate: **{global_kpis['cancel_rate']:.2%}**
- Delay medio: **{global_kpis['avg_delay_min']:.2f} min** | P95: **{global_kpis['p95_delay_min']:.2f} min**
- Ocupación media: **{global_kpis['avg_occupancy']:.3f}**
- Margen medio: **{global_kpis['avg_margin']:.2f}** | Margen total: **{global_kpis['total_margin']:.2f}**

## Puntualidad
- OTR ≤ 5 min: **{global_kpis['otr_5']:.2%}**
- OTR ≤ 15 min: **{global_kpis['otr_15']:.2%}**

## Top rutas por retraso medio
{top5.to_markdown(index=False)}

## Semáforo de rutas
- Rutas prioridad: **{', '.join(amber_routes) if amber_routes else 'N/A'}**
- Rutas saludables: **{', '.join(green_routes) if green_routes else 'N/A'}**

## Recomendación ejecutiva
Revisar semanalmente OTR≤15, delay medio, P95 y causas de disrupción. Priorizar rutas AMBER y ventanas horarias con retraso sistemático.
'''

(REPORTS / 'executive_summary.md').write_text(summary, encoding='utf-8')
print(summary)
